In [1]:
from model import Round
from sqlalchemy import select
from sqlalchemy.orm import Session
from sqlalchemy import create_engine
from collections import defaultdict

import toml
import numpy as np

# CALL gets_drilled() ON /turf/simulated/mineral

In [2]:
config = toml.load(open('config.toml'))
connection_string = config['database']['connection_string']
engine = create_engine(connection_string)

In [2]:
config = toml.load(open('config.toml'))
connection_string = config['database']['prod_connection_string']
engine = create_engine(connection_string)

In [8]:
prod_rounds = [36_666, 36_667, 36_668]

In [16]:
count = 0
with Session(engine) as session:
    rounds = session.query(Round).filter(Round.id.in_(prod_rounds))
    for round in rounds.all():
        count += 1
print(count)

3


In [6]:
count = 0
with Session(engine) as session:
    rounds = session.query(Round).all()
    for round in rounds:
        count += 1

In [7]:
count

2357

In [11]:
head_average_total_ore_counts = defaultdict(int)
head_local_test_rounds = (567, 568)

In [7]:
LATEST_ORE_VEIN_ROUND = 677

In [4]:
def percent_diff(v1, v2):
    return (abs(v1 - v2)) / ((v1 + v2)/2)

In [ ]:
rounds = [

In [3]:
def get_aggregate_stats(round_ids=None):
    stats = defaultdict(list)
    with Session(engine) as session:
        rounds = None
        if round_ids:
            rounds = session.query(Round).filter(Round.id.in_(round_ids))
        else:
            rounds = session.query(Round)
        for round in rounds.all():
            if round and round.has_feedback_stat('ore_mined'):
                ore_mined = round.get_feedback_stat('ore_mined')
                for ore, count in ore_mined.items():
                    stats[ore].append(count)

    return stats

def average_stats(stats):
    avg_stats = dict()
    for ore, counts in stats.items():
        count = len(counts)
        average = sum(counts) / count
        avg_stats[ore] = int(average)

    return avg_stats
    
def print_stats(ore_mined_stat):
    keys = sorted(ore_mined_stat)
    for key in keys:
        print(f"{key} - {ore_mined_stat[key]}")

In [8]:
latest_ore_vein_stats = defaultdict(int)
with Session(engine) as session:
    aggregate_stats = get_aggregate_stats([LATEST_ORE_VEIN_ROUND])
    latest_ore_vein_stats = average_stats(aggregate_stats)
    print_stats(latest_ore_vein_stats)

/obj/item/stack/ore/bluespace_crystal - 98
/obj/item/stack/ore/diamond - 110
/obj/item/stack/ore/glass/basalt/ancient - 744
/obj/item/stack/ore/gold - 594
/obj/item/stack/ore/iron - 6206
/obj/item/stack/ore/plasma - 2424
/obj/item/stack/ore/silver - 1914
/obj/item/stack/ore/titanium - 648
/obj/item/stack/ore/uranium - 2047


In [14]:
stats_with_ruins = defaultdict(int)
with Session(engine) as session:
    aggregate_stats = get_aggregate_stats(head_local_test_rounds)
    stats_with_ruins = average_stats(aggregate_stats)
    print_stats(stats_with_ruins)

/obj/item/stack/ore/bluespace_crystal - 126
/obj/item/stack/ore/diamond - 131
/obj/item/stack/ore/glass/basalt/ancient - 772
/obj/item/stack/ore/gold - 1119
/obj/item/stack/ore/iron - 9505
/obj/item/stack/ore/plasma - 2468
/obj/item/stack/ore/silver - 1509
/obj/item/stack/ore/titanium - 1260
/obj/item/stack/ore/uranium - 572


In [15]:
stats_with_no_ruins = defaultdict(int)
with Session(engine) as session:
    aggregate_stats = get_aggregate_stats([570])
    stats_with_no_ruins = average_stats(aggregate_stats)
    print_stats(stats_with_no_ruins)

/obj/item/stack/ore/bluespace_crystal - 129
/obj/item/stack/ore/diamond - 136
/obj/item/stack/ore/glass/basalt/ancient - 750
/obj/item/stack/ore/gold - 1132
/obj/item/stack/ore/iron - 9440
/obj/item/stack/ore/plasma - 2645
/obj/item/stack/ore/silver - 1457
/obj/item/stack/ore/titanium - 1178
/obj/item/stack/ore/uranium - 576


In [18]:
percent_diffs

{'/obj/item/stack/ore/plasma': 0.06923528261294738,
 '/obj/item/stack/ore/bluespace_crystal': 0.023529411764705882,
 '/obj/item/stack/ore/iron': 0.006861968857218264,
 '/obj/item/stack/ore/silver': 0.03506405933917734,
 '/obj/item/stack/ore/titanium': 0.06726825266611977,
 '/obj/item/stack/ore/glass/basalt/ancient': 0.02890932982917214,
 '/obj/item/stack/ore/uranium': 0.006968641114982578,
 '/obj/item/stack/ore/diamond': 0.03745318352059925,
 '/obj/item/stack/ore/gold': 0.011550422034651266}

In [27]:
percent_diffs = dict()
for k in set(list(stats_with_ruins.keys())).union(list(stats_with_no_ruins.keys())):
    percent_diffs[k] = percent_diff(stats_with_ruins.get(k, 0), stats_with_no_ruins.get(k, 0))

for k, v in percent_diffs.items():
    print(f"{k}: {v:.1%} or {int(stats_with_no_ruins[k] * v)} minerals")

/obj/item/stack/ore/plasma: 6.9% or 183 minerals
/obj/item/stack/ore/bluespace_crystal: 2.4% or 3 minerals
/obj/item/stack/ore/iron: 0.7% or 64 minerals
/obj/item/stack/ore/silver: 3.5% or 51 minerals
/obj/item/stack/ore/titanium: 6.7% or 79 minerals
/obj/item/stack/ore/glass/basalt/ancient: 2.9% or 21 minerals
/obj/item/stack/ore/uranium: 0.7% or 4 minerals
/obj/item/stack/ore/diamond: 3.7% or 5 minerals
/obj/item/stack/ore/gold: 1.2% or 13 minerals


In [28]:
set(list(stats_with_no_ruins.keys()))

{'/obj/item/stack/ore/bluespace_crystal',
 '/obj/item/stack/ore/diamond',
 '/obj/item/stack/ore/glass/basalt/ancient',
 '/obj/item/stack/ore/gold',
 '/obj/item/stack/ore/iron',
 '/obj/item/stack/ore/plasma',
 '/obj/item/stack/ore/silver',
 '/obj/item/stack/ore/titanium',
 '/obj/item/stack/ore/uranium'}

In [4]:
from collections import defaultdict

config = toml.load(open('config.toml'))
connection_string = config['database']['prod_connection_string']
engine = create_engine(connection_string)

r_count = 0
total_count = 0

prod_stats = get_aggregate_stats()

In [5]:
prod_stats

defaultdict(list,
            {'/obj/item/stack/ore/silver': [523,
              170,
              180,
              232,
              207,
              70,
              650,
              116,
              200,
              270,
              25,
              447,
              609,
              439,
              604,
              450,
              420,
              458,
              43,
              584,
              410,
              107,
              115,
              60,
              374,
              225,
              368,
              533,
              335,
              200,
              288,
              292,
              253,
              271,
              438,
              765,
              381,
              258,
              151,
              406,
              194,
              302,
              390,
              194,
              357,
              531,
              61,
              849,
              596,
              185,
       

In [6]:
prod_avg_stats = average_stats(prod_stats)

In [7]:
print_stats(prod_avg_stats)

/obj/item/stack/ore/bananium - 13
/obj/item/stack/ore/bluespace_crystal - 35
/obj/item/stack/ore/diamond - 36
/obj/item/stack/ore/glass/basalt/ancient - 205
/obj/item/stack/ore/gold - 293
/obj/item/stack/ore/iron - 1819
/obj/item/stack/ore/plasma - 641
/obj/item/stack/ore/silver - 323
/obj/item/stack/ore/titanium - 296
/obj/item/stack/ore/tranquillite - 4
/obj/item/stack/ore/uranium - 163
